# 05 — Label Generation and Feature Provenance Investigation

**Dataset:** `metaverse_transactions_dataset(1).csv`  
**Rows/columns expected:** 78,600 × 14  
**Target candidate:** `anomaly`  
**Reported origin:** Kaggle (exact Kaggle URL, dataset owner, version, license, and documentation still need to be recorded).

## Objective

Investigate whether the target label `anomaly` is independently supported by the observed fields, or whether it appears to have been assigned using `risk_score` or other feature rules.

This is a **forensic diagnostic notebook**, not a model-training notebook. It does not modify the original CSV, create new labels, or claim to recover the original author's code. Statistical association can identify suspicious patterns, but only the Kaggle dataset documentation, source code, or creator's explanation can establish the actual label-generation process.

## Important

- Keep the original CSV unchanged.
- Do not treat the Kaggle origin alone as proof that the data are real-world or independently labelled.
- Do not use `risk_score` as a predictive feature until its provenance and availability at prediction time are established.
- Complete the provenance decision register near the end of this notebook.

## 1. Load source and record provenance

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np
from IPython.display import display

DATA_PATH = Path("metaverse_transactions_dataset(1).csv")
if not DATA_PATH.exists():
    csvs = list(Path(".").glob("*.csv"))
    raise FileNotFoundError(
        f"Cannot find {DATA_PATH}. Upload the original CSV to this Colab session "
        f"or edit DATA_PATH. CSV files found: {[p.name for p in csvs]}"
    )

# Record the source file hash to identify the exact file version used.
sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
df = pd.read_csv(DATA_PATH)

print("File:", DATA_PATH.name)
print("SHA-256:", sha256)
print("Shape:", df.shape)
display(df.head())

### Kaggle source record — fill these fields from the Kaggle page

The user has identified Kaggle as the source, but the exact dataset page has not yet been supplied. Do not invent these details.

- Kaggle dataset URL: **TO FILL**
- Dataset title as shown on Kaggle: **TO FILL**
- Owner / uploader: **TO FILL**
- Version or update date: **TO FILL**
- License / usage terms: **TO FILL**
- Kaggle description and stated data-generation method: **TO FILL**
- Original source of transactions (if stated): **TO FILL**
- Does the page provide label definitions or generation code? **TO FILL**

Save a copy or screenshot of the Kaggle description, data dictionary, and any linked code/notebook. The Kaggle page's wording should be preserved rather than paraphrased as established fact.

## 2. Confirm schema, target, and basic integrity

In [ ]:
TARGET = "anomaly"
expected = [
    "timestamp", "hour_of_day", "sending_address", "receiving_address",
    "amount", "transaction_type", "location_region", "ip_prefix",
    "login_frequency", "session_duration", "purchase_pattern",
    "age_group", "risk_score", "anomaly"
]
print("Missing expected columns:", sorted(set(expected) - set(df.columns)))
print("Unexpected columns:", sorted(set(df.columns) - set(expected)))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Missing target labels:", int(df[TARGET].isna().sum()))
display(df[TARGET].value_counts(dropna=False).rename_axis(TARGET).to_frame("count"))
display((df[TARGET].value_counts(normalize=True, dropna=False)*100).round(3).rename("percent").to_frame())

## 3. Test whether the target is a deterministic function of risk_score

In [ ]:
if "risk_score" in df.columns:
    print("Unique risk_score values:", df["risk_score"].nunique())
    display(df.groupby(TARGET)["risk_score"].agg(
        count="size", minimum="min", q01=lambda s: s.quantile(.01),
        median="median", q99=lambda s: s.quantile(.99), maximum="max",
        unique_values="nunique"
    ))
    # Check whether score ranges for classes overlap.
    score_ranges = df.groupby(TARGET)["risk_score"].agg(["min", "max"])
    display(score_ranges)
    classes = list(score_ranges.index)
    overlap_rows = []
    for i, a in enumerate(classes):
        for b in classes[i+1:]:
            lo = max(score_ranges.loc[a, "min"], score_ranges.loc[b, "min"])
            hi = min(score_ranges.loc[a, "max"], score_ranges.loc[b, "max"])
            overlap_rows.append({"class_a": a, "class_b": b,
                                 "range_overlap": bool(lo <= hi),
                                 "overlap_lower": lo if lo <= hi else np.nan,
                                 "overlap_upper": hi if lo <= hi else np.nan})
    display(pd.DataFrame(overlap_rows))
    print("Unique labels within each risk_score value:")
    score_label_counts = df.groupby("risk_score")[TARGET].nunique()
    print("Risk-score values associated with >1 target class:",
          int((score_label_counts > 1).sum()))
    print("Risk-score values associated with exactly 1 target class:",
          int((score_label_counts == 1).sum()))

**Interpretation:** if each score value maps to exactly one target class and the class score ranges do not overlap, the target is perfectly recoverable from `risk_score` in this dataset. This is evidence of deterministic alignment, not proof of which variable was created first. The score might have generated the label, the label might have generated the score, or both might have been assigned from a third rule.

## 4. Examine categorical features against the target

In [ ]:
categorical_cols = [
    c for c in ["transaction_type", "location_region", "purchase_pattern", "age_group", "ip_prefix"]
    if c in df.columns
]
for col in categorical_cols:
    print(f"\\n### {col}")
    counts = pd.crosstab(df[col].fillna("<MISSING>").astype(str),
                         df[TARGET].fillna("<MISSING>").astype(str))
    row_pct = pd.crosstab(df[col].fillna("<MISSING>").astype(str),
                          df[TARGET].fillna("<MISSING>").astype(str),
                          normalize="index") * 100
    display(counts)
    display(row_pct.round(2))
    print("Distinct target classes per feature value:")
    display(df.groupby(col, dropna=False)[TARGET].nunique().rename("n_target_classes").to_frame())

## 5. Investigate relationships among suspected co-generated fields

In [ ]:
pairs = [
    ("age_group", "purchase_pattern"),
    ("transaction_type", "anomaly"),
    ("purchase_pattern", "anomaly"),
    ("age_group", "anomaly"),
]
for a, b in pairs:
    if a in df.columns and b in df.columns:
        print(f"\\n### {a} versus {b}")
        display(pd.crosstab(df[a], df[b], margins=True))
        if a != TARGET and b != TARGET:
            print("Unique values of", b, "within each", a)
            display(df.groupby(a, dropna=False)[b].nunique().rename("n_unique").to_frame())

## 6. Check numerical features and class-conditional patterns

In [ ]:
numeric_cols = [c for c in df.select_dtypes(include=np.number).columns if c != TARGET]
summary_tables = {}
for col in numeric_cols:
    print(f"\\n### {col}")
    tab = df.groupby(TARGET)[col].agg(["count", "min", "median", "mean", "max", "std"])
    display(tab)
    summary_tables[col] = tab

## 7. Check whether engineered-looking fields can be reconstructed

A field that can be reconstructed exactly from another column may be redundant. If it is also associated with the target, it may be a proxy for a label rule. This section tests only explicit, observable relationships; it does not infer hidden generation code.

In [ ]:
# Timestamp -> hour_of_day consistency
if "timestamp" in df.columns and "hour_of_day" in df.columns:
    parsed_ts = pd.to_datetime(df["timestamp"], errors="coerce")
    derived_hour = parsed_ts.dt.hour
    valid = parsed_ts.notna()
    print("Invalid timestamps:", int((~valid).sum()))
    print("hour_of_day mismatches:", int((df.loc[valid, "hour_of_day"].to_numpy() !=
                                           derived_hour.loc[valid].to_numpy()).sum()))

# Address self-transfers and repeated timestamps
if {"sending_address", "receiving_address"}.issubset(df.columns):
    self_transfer = df["sending_address"].astype(str).str.lower() == df["receiving_address"].astype(str).str.lower()
    print("Rows with same sender and receiver:", int(self_transfer.sum()))

if "timestamp" in df.columns:
    print("Repeated timestamp rows (beyond first occurrence):",
          int(df["timestamp"].duplicated(keep="first").sum()))

# Inspect ip_prefix values and type; values like 192.0 / 172.0 may be truncated prefixes,
# not complete IP addresses. Confirm from source documentation.
if "ip_prefix" in df.columns:
    print("ip_prefix dtype:", df["ip_prefix"].dtype)
    print("ip_prefix unique values:", sorted(df["ip_prefix"].dropna().unique().tolist()))
    print("ip_prefix value counts:")
    display(df["ip_prefix"].value_counts(dropna=False).to_frame("count"))

## 8. Check entity repetition and potential split contamination

In [ ]:
for col in ["sending_address", "receiving_address"]:
    if col in df.columns:
        vc = df[col].value_counts(dropna=False)
        print(f"\\n{col}: {df[col].nunique(dropna=False)} unique values")
        print("Values appearing more than once:", int((vc > 1).sum()))
        print("Rows belonging to repeated values:", int(vc[vc > 1].sum()))
        display(vc.head(10).rename("row_count").to_frame())

**Interpretation:** repeated addresses can create entity overlap between train and test sets. This is not automatically leakage, but a random row split may overstate generalization to unseen wallets. For a future model, decide whether the intended deployment predicts future activity from known entities or generalizes to unseen entities. Use chronological and/or group-aware evaluation to match that deployment scenario.

## 9. Create an evidence-based feature risk register

In [ ]:
# This is a starting register, not an automatic final feature selection.
register = pd.DataFrame([
    ["risk_score", "Very high", "Perfect target alignment observed in earlier audit; score range separated classes", "Exclude from predictive inputs unless independent pre-label provenance is proven"],
    ["transaction_type", "High", "Strong target association observed; exact class-specific pattern requires review", "Hold out pending source/label rule verification"],
    ["purchase_pattern", "High", "Appears linked to age_group and target categories", "Hold out pending provenance verification"],
    ["age_group", "High", "Appears linked to purchase_pattern and target categories", "Hold out pending provenance verification"],
    ["login_frequency", "Medium / unknown", "Strong association observed in prior single-feature diagnostic", "Investigate generation process and prediction-time availability"],
    ["session_duration", "Medium / unknown", "Strong association observed in prior single-feature diagnostic", "Investigate generation process and prediction-time availability"],
    ["timestamp", "Unknown", "Temporal field; potential leakage if label reflects future outcomes or time-specific generation", "Use for chronology/splitting; derive permitted time features carefully"],
    ["hour_of_day", "Unknown", "Appears derivable from timestamp", "Check redundancy and prediction-time availability"],
    ["sending_address", "Identifier risk", "Repeated entity identifier", "Do not treat as ordinary numeric/categorical signal without justification; consider group split"],
    ["receiving_address", "Identifier risk", "Repeated entity identifier", "Do not treat as ordinary numeric/categorical signal without justification; consider group split"],
    ["amount", "Unknown", "Numeric transaction field", "Verify units, generation, and timing"],
    ["location_region", "Unknown", "Categorical context field", "Verify provenance and whether it is independently observed"],
    ["ip_prefix", "Unknown", "Stored as float with few unique values", "Verify representation and semantics from source documentation"],
], columns=["feature", "provisional_risk", "observed_reason", "provisional_action"])
display(register)
register.to_csv("05_feature_provenance_risk_register.csv", index=False)
print("Saved 05_feature_provenance_risk_register.csv")

## 10. What this notebook can and cannot establish

### It can establish from the CSV
- Whether target classes align perfectly or near-perfectly with observed fields.
- Whether categorical fields have deterministic or near-deterministic mappings.
- Whether one field is reconstructible from another.
- Whether entity identifiers repeat and may affect evaluation design.
- Which features need source documentation before model use.

### It cannot establish from the CSV alone
- The original Kaggle creator's actual label-generation code.
- Whether the data are synthetic, simulated, anonymized, or collected from real transactions.
- Whether a field was available at the time a real prediction would have been made.
- Whether a strong association is causal or merely correlated.
- Whether the original labels represent independently verified fraud/anomaly outcomes.

To establish those points, obtain the exact Kaggle dataset URL, dataset description, data dictionary, linked notebooks/scripts, and any cited source paper.

## 11. Export a reproducibility summary

In [ ]:
from pathlib import Path
summary = {
    "source_file": DATA_PATH.name,
    "sha256": sha256,
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "target_column": TARGET,
    "target_counts": {str(k): int(v) for k, v in df[TARGET].value_counts(dropna=False).items()},
    "missing_values_total": int(df.isna().sum().sum()),
    "exact_duplicate_rows": int(df.duplicated().sum()),
    "kaggle_url": "TO FILL",
    "kaggle_owner": "TO FILL",
    "kaggle_version": "TO FILL",
    "license": "TO FILL",
}
Path("05_provenance_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("Saved 05_provenance_summary.json")

## 12. Final decision gate

Do **not** proceed to a final ECIPS predictive-model claim until the following are completed:

- [ ] Exact Kaggle URL and uploader recorded.
- [ ] Kaggle description and license saved.
- [ ] Label definition for each `anomaly` class found or explicitly marked undocumented.
- [ ] `risk_score` generation rule found or explicitly marked unknown.
- [ ] Provenance of `transaction_type`, `purchase_pattern`, `age_group`, `login_frequency`, and `session_duration` checked.
- [ ] Prediction-time availability documented for every retained feature.
- [ ] Evaluation split selected to match intended deployment (future transactions, unseen entities, or both).
- [ ] Suspicious fields excluded or justified with documentary evidence.

**Research integrity note:** if the Kaggle page provides no label-generation documentation, report that limitation transparently. Do not claim that labels are ground-truth fraud labels unless the source supports that statement.